In [ ]:
# Install the fastai library with quiet output
!pip install -Uqq fastai

# Import all vision-related functions and classes from fastai
from fastai.vision.all import *

# Import FP16 (mixed precision) callback for memory optimization during training
from fastai.callback.fp16 import *

# Import matplotlib for plotting and visualization
import matplotlib.pyplot as plt

# Import NumPy for numerical operations and array handling
import numpy as np

# Import partial from functools for creating partial functions
from functools import partial

# Import random number generation functions from NumPy
from numpy.random import normal, seed, uniform

# Import pandas for data manipulation and analysis
import pandas as pd

# Import functional operations from PyTorch for neural network operations
import torch.nn.functional as F

# Import neural network modules from PyTorch
import torch.nn as nn

# Import PyTorch for deep learning operations and tensor computations
import torch

# Import regular expressions module for pattern matching and string manipulation
import re

In [ ]:
# Download and extract the PETS dataset from fastai URLs
path = untar_data(URLs.PETS)

# List all images in the extracted dataset directory
(path/"images").ls()

In [ ]:
# Define a DataBlock that specifies the structure for image classification
# blocks: (ImageBlock, CategoryBlock) - images as input, categories as labels
pets = DataBlock(blocks = (ImageBlock, CategoryBlock),
                 # get_items: Use get_image_files function to find all images
                 get_items=get_image_files,
                 # splitter: Randomly split data with seed 42 for reproducibility
                 splitter=RandomSplitter(seed=42),
                 # get_y: Extract label from filename using regex pattern and name attribute
                 get_y=using_attr(RegexLabeller(r'(.+)_\d+\.[^.]+$'), 'name'),
                 # item_tfms: Resize all images to 460x460 pixels
                 item_tfms=Resize(460),
                 # batch_tfms: Apply augmentation transforms and resize to 224x224 with minimum scale 0.75
                 batch_tfms=aug_transforms(size=224, min_scale=0.75))

# Create DataLoaders from the DataBlock for the images directory
dls = pets.dataloaders(path/"images")

# Display a sample batch of images (1 row, 3 columns) to visualize the data
dls.show_batch(nrows = 1, ncols = 3)

In [ ]:
# Create a vision learner using ResNet34 architecture with error rate as the metric
learn = vision_learner(dls, resnet34, metrics=error_rate)

# Find the optimal learning rate by training with varying learning rates
lr = learn.lr_find()

# Fine-tune the model for 6 epochs using the optimal learning rate found above
learn.fine_tune(6, base_lr=lr.valley)

In [ ]:
# Create a second vision learner using ResNet34 architecture with error rate as the metric
learn2 = vision_learner(dls, resnet34, metrics=error_rate)

# Find the optimal learning rate for learn2
lr2 = learn2.lr_find()

# Train learn2 for 3 epochs using one-cycle learning rate schedule with optimal lr
learn2.fit_one_cycle(3, lr2.valley)

# Unfreeze the model 
learn2.unfreeze()

# Find a new learning rate after unfreezing the model
lr2_new = learn2.lr_find()

# Train the unfrozen model for 6 more epochs with the new learning rate
learn2.fit_one_cycle(6, lr2_new.valley)

In [ ]:
# Create a third vision learner using ResNet34 architecture with error rate as the metric
learn3 = vision_learner(dls, resnet34, metrics=error_rate)

# Find the optimal learning rate for learn3
lr3 = learn3.lr_find()

# Train learn3 for 3 epochs using one-cycle learning rate schedule with optimal lr
learn3.fit_one_cycle(3, lr3.valley)

# Unfreeze the model to allow all layers to be trainable
learn3.unfreeze()

# Find a new learning rate after unfreezing the model
lr3_new = learn3.lr_find()

# Train the unfrozen model for 12 epochs using discriminative learning rates (1e-6 for early layers, lr3_new.valley for later layers)
learn3.fit_one_cycle(12, lr_max=slice(1e-6, lr3_new.valley))

In [ ]:
# Plot the training and validation loss curve for the first learner (learn)
learn.recorder.plot_loss()

# Plot the training and validation loss curve for the second learner (learn2)
learn2.recorder.plot_loss()

# Plot the training and validation loss curve for the third learner (learn3)
learn3.recorder.plot_loss()